# Bispectrum and bicoherence

## Summary

Bicoherence is a squared, normalized version of the bispectrum. Bicoherence is real and normalized between 0 and 1. Bicoherence measure how much phase coupling / "locking" there is between signals. In the following, we consider auto-bispectrum and auto-bicoherence - i.e. the phase coupling between different frequency components in the same signal.

## Definitions

### Bispectrum

$B(f_1, f_2) = F(f_1) . F(f_2) . F^*(f_1+f_2)$  , with $f_1$ and $f_2$ the 2 frequencies considered, $B$ the bispectrum, $F$ the Fourier transform, $^*$ the complex conjugate.

A $F$ transform results in a complex vector. From standard complex calculations, the $B$ magnitude is the product of the $F$ magnitudes, and the $B$ phase is the sum of the $F$ phases (with a - for the $^*$ one).

## Bicoherence

The bicoherence is obtained by averaging over several bispectra, and normalizing. There are several ways to do the normalization. We will prefer the "absolute norm" normalization:

$b(f_1, f_2) = abs(sum_n(B_n(f_1, f_2)) / sum_n(abs(B_n(f_1, f_2)))$  , with $b$ the bicoherence, $B_n(f_1, f_2)$ the bicoherence spectrum on segment number $n$ at frequencies $f_1$ and $f_2$, $sum_n$ the sum over the $n$ segments.

### Interpretation

Considering a signal that is stationary in time:

- if the signal components at frequencies $f_1$, $f_2$, $f_3$, are not phase locked, then the phase of the $B_n(f_1, f_2)$ will be random, and the $sum_n(B_n(f_1, f_2)$ will average to $0$ at the limit; hence, the bicoherence will be $0$
- if the signal components at frequencies $f_1$, $f_2$, $f_3$ are phase locked, then the phase of the $B_n(f_1, f_2)$ will be constant, and the $abs(sum_n(B_n(f_1, f_2))$ will average to $n . abs(B_n(f_1, f_2))$; so does the denominator, so the bicoherence will be $1$.

So a bicoherence of $0$ indicates that the signals at $f_1$, $f_2$, $f_3$ are not phase locked / coupled, and a bicoherence of $1$ indicate that they are fully phase locked / coupled.

Note that this is a necessary, but not a sufficient, condition for frequency components to be coupled by a non-linear mechanism (like "correlation is not causation").

## Sources

- https://en.wikipedia.org/wiki/Bicoherence

In [41]:
import numpy as np
import numpy.typing as npt

from scipy import fftpack

In [88]:
# A verbose, slow, un-optimized implementation of real-signal bispectrum and bi-coherence.
# The goal is correctness and ease-of-reading, not speed!
# Still, we use FFTs in reasonable ways! :)


def split_signal_into_segments(signal: npt.NDArray, segment_length: int, n_overlap: int, use_next_fftlength: bool =True) ->  npt.NDArray:
    """Split a signal into segments:
    Arguments:
        - signal: the signal to split, a 1d numpy array
        - segment_length: the (minimum if use_next_fftlength is True) segment length
        - overlap: the int number of overlap samples between 2 consecutive segments
        - use_next_fftlength: True if should use the next segment length that allows fast FFT, False to force the current segment length
    Returns:
        - array_of_signals: such that array_of_signals[0, :] is the first segment of length segment_length, etc"""

    assert isinstance(signal, np.ndarray)
    assert signal.ndim == 1
    assert isinstance(segment_length, int)
    assert isinstance(n_overlap, int)
    assert isinstance (use_next_fftlength, bool)
    
    if use_next_fftlength:
        segment_length = fftpack.next_fast_len(segment_length)

    start_segments = np.arange(0, len(signal)-segment_length, segment_length-n_overlap)
    nbr_segments = len(start_segments)
    array_of_signals = np.full([nbr_segments, segment_length], np.nan)

    for crrt_segment in range(nbr_segments):
        array_of_signals[crrt_segment, :] = signal[start_segments[crrt_segment]: start_segments[crrt_segment]+segment_length]
    
    return array_of_signals


array_of_signals = split_signal_into_segments(np.arange(0, 10, 1), segment_length=4, n_overlap=2, use_next_fftlength=True)
res = np.array(
    [[0., 1., 2., 3.],
     [2., 3., 4., 5.],
     [4., 5., 6., 7.]])
np.testing.assert_allclose(res, array_of_signals)
array_of_signals = split_signal_into_segments(np.arange(0, 12, 1), segment_length=7, n_overlap=5, use_next_fftlength=True)
res = np.array(
    [[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
     [ 3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]])
np.testing.assert_allclose(res, array_of_signals)


def find_first_greater_or_equal_index(value:float, array: npt.NDArray):
    """Find the index of the first element in the array that is greater than the specified value.
    Arguments:
        - array (numpy.ndarray): The array to search.
        - value (float): The value to compare against.
    Returns:
        - int: The index of the first element greater than the specified value, the last index."""

    bool_array = array > value
    if bool_array.any():
        argmax = np.argmax(bool_array)
        return argmax
    else:
        return len(array)-1


def get_f_range_index(f_range, frequencies: npt.NDArray):
    """Get the range of index in frequencies that cover f_range. We assume that frequencies is an array of rfft
    frequencies from rfftfreq. We always skip the 0-frequency and the last frequency.
    Arguments:
        - f_range: the range of frequencies to cover, either [f1, f2] with f1<f2 and both are floats, or None
        - frequencies: the array of frequencies from rfftfreq
    Returns:
        - (idx_min, idx_max): the range of indexes so that frequencies[idx_min:idx_max] cover f_range"""

    assert isinstance(frequencies, np.ndarray)
    assert frequencies.ndim == 1
    np.testing.assert_approx_equal(0.0, frequencies[0])
    
    if f_range is None:
        return (1, len(frequencies)-2)
    else:
        assert f_range[0] < f_range[1]
        idx_min = find_first_greater_index(f_range[0], frequencies)
        if idx_min > 1:
            idx_min = idx_min - 1
        idx_min = max(1, idx_min)
        idx_max = find_first_greater_index(f_range[1], frequencies)
        if idx_max < len(frequencies)-2:
            idx_max = idx_max + 1
        idx_max = min(idx_max, len(frequencies)-2)
        return (int(idx_min), int(idx_max))


assert get_f_range_index(None, np.array([0, 1, 2, 3, 4, 5])) == (1, 4)
assert get_f_range_index([-1.0, 7.0], np.array([0, 1, 2, 3, 4, 5])) == (1, 4)
assert get_f_range_index([2, 3], np.array([0, 1, 2, 3, 4, 5])) == (2, 4)

def compute_auto_bispectrum(signal: npt.NDArray, sample_frequency: float, window=np.hanning, f1_range=None, f2_range=None):
    """Compute a single bispectrum for a real signal.
    Arguments:
        - signal: the signal on which to compute the bispectrum
        - sample_frequency: the sampling frequency
        - window: the windowing algorithm, a window function from
            https://numpy.org/doc/stable/reference/routines.window.html ,
            or None for no window
        - f1_range: the range [min_f1, max_f1] over which take the bisepctrum; if None use all frequencies
        - f2_range: the range [min_f1, max_f2] over which take the bispectrum; if None use all frequencies
    Returns:
        - frequencies: the frequencies of the bispectrum
        - bispectrum: the bispectrum as a np array"""
    
    assert isinstance(signal, np.ndarray)
    assert signal.ndim == 1
    assert np.issubdtype(array.dtype, np.floating) or np.issubdtype(array.dtype, np.integer)
    assert isinstance(fs, float)
    assert fs > 0.0

    if window is not None:
        window = window(len(signal))
        signal = signal * window

    rfft = np.fft.rfft(signal)
    frequencies = np.fft.rfftfreq(len(signal), 1.0/fs)

    (min_f1, max_f1) = get_f_range_index(f1_range, frequencies)
    (min_f2, max_f2) = get_f_range_index(f2_range, frequencies)

    # TODO: array of indexes to add in match
    # TODO: then do all in a single np operation
    
    return frequencies, bispectrum

def compute_auto_bispectra_on_array_of_signals(array_of_signals, sample_frequency, window=np.hanning):
    return frequencies, array_of_auto_bispectra

def compute_auto_bicoherence_on_array_of_auto_bispectra(frequencies, array_of_auto_bispectra, method="absolute_norm"):
    return frequencies, auto_bicoherence